# Hardy–Weinberg and Wright–Fisher genetic drift

*bioinformatics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/wright_fisher_drift)

Random mating gives the Hardy–Weinberg genotype frequencies $p^2, 2pq, q^2$; random sampling of $2N$ gene copies each generation makes allele frequencies drift. Thousands of populations evolve at once as one (replicates, generations) tensor of binomial draws, optionally with selection $s$, and are compared with $(1-1/2N)^t$ for heterozygosity and with $1/2N$ and Kimura's formula for fixation.

**Vector form**

$$f_{AA}, f_{Aa}, f_{aa} = p^2,\ 2pq,\ q^2,\qquad 2N\,p_{t+1} \sim \mathrm{Bin}\!\Big(2N,\ \frac{p_t(1+s)}{1+s\,p_t}\Big),\qquad \mathbb E[H_t] = H_0\Big(1-\frac{1}{2N}\Big)^t,\qquad u(p_0) = \frac{1-e^{-4Nsp_0}}{1-e^{-4Ns}}$$

**Index (tensor) form**

$$G_{rg} = \big(p_r^2,\ 2p_rq_r,\ q_r^2\big)_g \qquad p'_{rt} = \frac{p_{rt}(1+s)}{1+s\,p_{rt}} \qquad p_{r,t+1} = \frac{K_{rt}}{2N},\ \ K_{rt}\sim\mathrm{Bin}(2N, p'_{rt}) \qquad \bar H_t = \frac1R\sum_r 2p_{rt}q_{rt}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $p_{rt}$ | `simulate(p0, N, s, G, gen)` | (R, G+1) | frequency of allele A in replicate population $r$ at generation $t$ |
| $p'_{rt}$ | `select(p, s)` | (R,) | after selection: A copies are passed on $1+s$ times as often |
| $K_{rt}$ | `torch.binomial(2N, p_sel)` | (R,) | number of A among the $2N$ copies of the next generation: one draw per population, all at once |
| $G_{rg}$ | `hardy_weinberg(p)` | (…, 3) | genotype frequencies AA, Aa, aa under random mating |
| $G^{\mathrm{obs}}_{rtg}$ | `sample_genotypes(p, N, gen)` | (K, G+1, 3) | $N$ individuals made of two random gametes each, counted by genotype (dots) |
| $\bar H_t$ | `heterozygosity(p).mean(dim=0)` | (G+1,) | sum over $r$: mean $2pq$, compared with $H_0(1-1/2N)^t$ |
| $u(s)$ | `(p_final == 1).mean(dim=1)` | (S,) | fraction of populations where a single new copy ($p_0 = 1/2N$) was fixed, for each $s$ |
| $u_{\mathrm{Kimura}}$ | `kimura_fixation(1/2N, N, s)` | (S,) | $(1-e^{-4Nsp_0})/(1-e^{-4Ns})$, equal to $p_0 = 1/2N$ when $s = 0$ |

Hardy–Weinberg says that one round of random mating turns allele frequency $p$ into genotype frequencies $p^2, 2pq, q^2$, whatever the genotypes of the parents were; the dots are $N$ individuals built from two random gametes and scatter around those curves. Allele frequencies stay put only in an infinite population. With $2N$ copies each generation is a binomial sample of the previous one, so $p$ performs a random walk until it hits 0 (allele lost) or 1 (allele fixed). Two copies picked at random share a parent copy with probability $1/2N$, so the expected heterozygosity $2pq$ shrinks by the factor $1 - 1/2N$ per generation; drift is faster in small populations. A new neutral mutation starts as one copy out of $2N$; since in the end one of the $2N$ copies will be the ancestor of all, its chance of fixation is $1/2N$. With selection Kimura's diffusion result applies: a beneficial allele with $4Ns \gg 1$ is fixed with probability close to $2s$, still far from certain, and a mildly harmful one can be fixed by drift when $|4Ns| \lesssim 1$. Here $N$ is the idealised population size (constant, random mating, no mutation or migration); real populations are described by a smaller effective size $N_e$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/wright_fisher_drift/main.py
"""Main loop: replicate populations → Wright–Fisher drift → heterozygosity, fixation, Hardy–Weinberg → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

N_SHOWN = 60  # populations whose genotypes are drawn in the Hardy–Weinberg panel


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    gen = torch.Generator(device=dev).manual_seed(params["seed"])
    N, s, G = params["N"], params["s"], params["generations"]
    R = 200 if fast_mode() else params["replicates"]
    n_fix = 2000 if fast_mode() else params["fix_replicates"]

    # 1) drift (with optional selection) in R populations that all start at p0
    p = solve.simulate(tensors.initial_frequencies(R, params["p0"], dev), N, s, G, gen)  # (R, G+1)
    t = torch.arange(G + 1, device=dev, dtype=torch.float64)
    H_sim = equation.heterozygosity(p).mean(dim=0)  # (G+1,) average over replicates
    H_theory = equation.expected_heterozygosity(H_sim[0].item(), N, t)  # (G+1,) neutral prediction

    # 2) fixation of a single new mutant (p0 = 1/2N) for several selection strengths
    s_grid = tensors.selection_grid(N, s, dev)  # (S,)
    u_sim = solve.fixation(tensors.new_mutants(len(s_grid), n_fix, N, dev), N, s_grid, gen, max_gen=40 * N)  # (S,)
    u_theory = equation.kimura_fixation(1 / (2 * N), N, s_grid)  # (S,)

    # 3) Hardy–Weinberg: genotypes of N individuals made from random gametes, in a few populations
    G_obs = solve.sample_genotypes(p[:N_SHOWN], N, gen)  # (K, G+1, 3) in every generation
    G_hw = equation.hardy_weinberg(p[:N_SHOWN, 0])  # (K, 3) expected at t = 0

    fixed, lost = (p[:, -1] == 1).double().mean().item(), (p[:, -1] == 0).double().mean().item()
    print(f"N = {N} diploids (2N = {2 * N} copies), s = {s:g}, {R} populations from p0 = {params['p0']:g}")
    print(f"Hardy–Weinberg at t = 0: AA, Aa, aa observed {_fmt(G_obs[:, 0].mean(0))}  expected {_fmt(G_hw.mean(0))}")
    for tt in sorted({G // 4, G // 2, G}):
        print(f"t = {tt:4d}: mean H = {H_sim[tt]:.4f}   H0 (1 - 1/2N)^t = {H_theory[tt]:.4f}")
    print(f"after {G} generations: fixed {fixed:.1%}, lost {lost:.1%}, still segregating {1 - fixed - lost:.1%}")
    print(f"single new mutant, {n_fix} populations per s:   1/2N = {1 / (2 * N):.4f}")
    for k in range(len(s_grid)):
        print(f"  4Ns = {4 * N * s_grid[k]:6.2f}   fixed {u_sim[k]:.4f}   Kimura {u_theory[k]:.4f}")
    return plot.animate(
        p, G_obs, H_sim, H_theory, s_grid, u_sim, u_theory, N, s, n_fix, out_dir / "wright_fisher_drift.gif"
    )


def _fmt(x: torch.Tensor) -> str:
    return "(" + ", ".join(f"{v:.3f}" for v in x.tolist()) + ")"


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/wright_fisher_drift/equation.py
r"""Population genetics → tensor translation.

One gene with two alleles A and a in a diploid population of N individuals (2N gene copies). p is the frequency of A.

Hardy–Weinberg: if gametes meet at random, the genotypes AA, Aa, aa occur with frequencies

    G_{r·} = (p_r², 2 p_r q_r, q_r²),      q = 1 - p

Wright–Fisher: every generation is 2N gene copies drawn at random from the previous one. With selection, A copies
are (1 + s) times as likely to be passed on, which first shifts the frequency to p' and then the drawing adds noise:

    p'_{rt}    = p_{rt} (1 + s) / (1 + s p_{rt})
    p_{r,t+1}  = K_{rt} / 2N,         K_{rt} ~ Binomial(2N, p'_{rt})

Index r labels independent replicate populations, t the generation; every replicate advances in one tensor call.
No index is summed in the update itself; averages over replicates (Σ_r / R) give the curves compared with theory:

    E[H_t] = H_0 (1 - 1/2N)^t,  H = 2pq        (neutral drift)
    u(p_0) = (1 - e^{-4Nsp_0}) / (1 - e^{-4Ns})  (Kimura: chance that A is eventually fixed; u = p_0 for s = 0)
"""

import torch


def hardy_weinberg(p: torch.Tensor) -> torch.Tensor:
    """G_{…g} = (p², 2pq, q²): genotype frequencies of AA, Aa, aa under random mating.   (…,) → (…, 3)"""
    q = 1 - p
    return torch.stack([p * p, 2 * p * q, q * q], dim=-1)


def heterozygosity(p: torch.Tensor) -> torch.Tensor:
    """H = 2pq: chance that two gene copies drawn at random differ.   same shape as p"""
    return 2 * p * (1 - p)


def select(p: torch.Tensor, s: float | torch.Tensor) -> torch.Tensor:
    """p' = p (1 + s) / (1 + s p): allele A has relative fitness 1 + s (per copy).   same shape as p"""
    return p * (1 + s) / (1 + s * p)


def wright_fisher_step(p: torch.Tensor, N: int, s: float | torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    """p_{t+1} = Binomial(2N, p'_t) / 2N for every replicate at once.   (…,) → (…,)"""
    p_sel = select(p, s)
    copies = torch.full_like(p_sel, 2 * N)
    return torch.binomial(copies, p_sel, generator=gen) / (2 * N)


def expected_heterozygosity(H0: float, N: int, t: torch.Tensor) -> torch.Tensor:
    """E[H_t] = H_0 (1 - 1/2N)^t: each generation two copies share a parent with probability 1/2N.   (T,)"""
    return H0 * (1 - 1 / (2 * N)) ** t


def kimura_fixation(p0: float, N: int, s: torch.Tensor) -> torch.Tensor:
    """u = (1 - e^{-4N s p0}) / (1 - e^{-4N s}), and u = p0 in the neutral limit s → 0.   (S,)"""
    x = 4 * N * s
    neutral = x.abs() < 1e-9
    safe = torch.where(neutral, torch.ones_like(x), x)  # avoid 0 / 0; replaced by p0 below
    u = torch.expm1(-safe * p0) / torch.expm1(-safe)
    return torch.where(neutral, torch.full_like(x, p0), u)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/wright_fisher_drift/tensors.py
"""Build the concrete tensors: starting allele frequencies of the replicate populations and the selection grid."""

import torch

# selection strengths for the fixation experiment, as the population-scaled 4Ns
SCALED_S = [-2.0, -1.0, 0.0, 1.0, 2.0, 4.0, 8.0]


def initial_frequencies(n_replicates: int, p0: float, device: torch.device) -> torch.Tensor:
    """p_{r,0} = p0 for every replicate population r.   (R,)"""
    return torch.full((n_replicates,), p0, device=device, dtype=torch.float64)


def new_mutants(n_s: int, n_replicates: int, N: int, device: torch.device) -> torch.Tensor:
    """A single new copy of A in each population: p = 1/2N, for every selection value and replicate.   (S, R)"""
    return torch.full((n_s, n_replicates), 1 / (2 * N), device=device, dtype=torch.float64)


def selection_grid(N: int, s_user: float, device: torch.device) -> torch.Tensor:
    """s values (S,) with 4Ns from SCALED_S, plus the slider value s if it is not already there."""
    s = [x / (4 * N) for x in SCALED_S]
    if all(abs(s_user - v) > 1e-12 for v in s):
        s.append(s_user)
    return torch.tensor(sorted(s), device=device, dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/wright_fisher_drift/solve.py
"""Run Wright–Fisher drift for many populations at once, the fixation experiment and Hardy–Weinberg genotype draws."""

import torch

from problems._common.progress import steps

from . import equation


def simulate(p0: torch.Tensor, N: int, s: float, generations: int, gen: torch.Generator) -> torch.Tensor:
    """Allele frequencies of every replicate in every generation.   p0 (R,) → (R, G+1)"""
    history = [p0]
    p = p0
    for _ in steps(generations, "generations"):
        p = equation.wright_fisher_step(p, N, s, gen)
        history.append(p)
    return torch.stack(history, dim=1)


def fixation(p0: torch.Tensor, N: int, s: torch.Tensor, gen: torch.Generator, max_gen: int) -> torch.Tensor:
    """Fraction of populations in which A is fixed, run until every population has fixed or lost it.

    p0 (S, R) with one row per selection value s (S,).  Returns (S,).
    """
    p = p0
    s_col = s[:, None]  # (S, 1) broadcast over replicates
    for _ in steps(max_gen, "until fixed or lost"):
        p = equation.wright_fisher_step(p, N, s_col, gen)
        if not ((p > 0) & (p < 1)).any():  # every population has fixed (p = 1) or lost (p = 0) the allele
            break
    return (p == 1).double().mean(dim=1)


def sample_genotypes(p: torch.Tensor, N: int, gen: torch.Generator) -> torch.Tensor:
    """Make N individuals per population from two random gametes each and count AA, Aa, aa.   (…,) → (…, 3) freqs."""
    gametes = torch.rand(*p.shape, N, 2, generator=gen, dtype=p.dtype, device=p.device) < p[..., None, None]
    copies = gametes.sum(dim=-1)  # (…, N) number of A copies of each individual: 0, 1 or 2
    return torch.stack([(copies == 2), (copies == 1), (copies == 0)], dim=-1).double().mean(dim=-2)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/wright_fisher_drift/plot.py
"""Animate drift: trajectories, the spread of allele frequencies, heterozygosity decay, Hardy–Weinberg, fixation."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import equation

FIXED, LOST, OPEN = "#2563eb", "#ea580c", "#9ca3af"
GENO = [("AA", "#2563eb"), ("Aa", "#16a34a"), ("aa", "#ea580c")]
N_LINES = 40  # trajectories drawn


def animate(
    p: torch.Tensor,
    G_obs: torch.Tensor,
    H_sim: torch.Tensor,
    H_theory: torch.Tensor,
    s_grid: torch.Tensor,
    u_sim: torch.Tensor,
    u_theory: torch.Tensor,
    N: int,
    s: float,
    n_fix: int,
    path: Path,
) -> Path:
    P = p.cpu().numpy()  # (R, G+1)
    geno = G_obs.cpu().numpy()  # (K, G+1, 3)
    H, H_th = H_sim.cpu().numpy(), H_theory.cpu().numpy()
    G = P.shape[1] - 1
    t = np.arange(G + 1)
    final = P[:N_LINES, -1]
    fate = np.where(final == 1, FIXED, np.where(final == 0, LOST, OPEN))

    fig = plt.figure(figsize=(9.6, 5.2))
    grid = fig.add_gridspec(2, 3, height_ratios=[1.1, 1])
    ax_p = fig.add_subplot(grid[0, :2])
    ax_hist = fig.add_subplot(grid[0, 2])
    ax_h = fig.add_subplot(grid[1, 0])
    ax_hw = fig.add_subplot(grid[1, 1])
    ax_u = fig.add_subplot(grid[1, 2])

    # --- allele-frequency trajectories, coloured by their fate at the end ---
    ax_p.set_xlim(0, G)
    ax_p.set_ylim(-0.02, 1.02)
    ax_p.set_xlabel("generation $t$", labelpad=1)
    ax_p.set_ylabel("frequency $p$ of A")
    ax_p.set_title(f"Wright–Fisher drift, N = {N}, s = {s:g}: blue fixed, orange lost", fontsize=10)
    lines = [ax_p.plot([], [], lw=0.9, color=fate[r])[0] for r in range(min(N_LINES, len(P)))]
    clock = ax_p.text(0.01, 0.93, "", transform=ax_p.transAxes, fontsize=9)

    # --- distribution of p over all replicates ---
    edges = np.linspace(0, 1, 22)
    ax_hist.set_xlim(0, 1)
    ax_hist.set_ylim(0, 1)
    ax_hist.set_xlabel("$p$", labelpad=1)
    ax_hist.set_ylabel("fraction of populations")
    ax_hist.set_title(f"all {len(P)} populations", fontsize=10)
    bars = ax_hist.bar(edges[:-1], np.zeros(21), width=edges[1] - edges[0], align="edge", color="#93c5fd", ec="#2563eb")
    fate_text = ax_hist.text(0.5, 0.9, "", transform=ax_hist.transAxes, ha="center", fontsize=8)

    # --- mean heterozygosity against the neutral prediction ---
    ax_h.plot(t, H_th / H[0], "--", color="#374151", lw=1.2, label=r"$(1 - 1/2N)^t$")
    h_line = ax_h.plot([], [], color="#dc2626", lw=1.6, label="simulation")[0]
    ax_h.set_xlim(0, G)
    ax_h.set_ylim(0, 1.05)
    ax_h.set_xlabel("generation $t$", labelpad=1)
    ax_h.set_title(r"heterozygosity $\langle 2pq\rangle / H_0$", fontsize=10)
    ax_h.legend(fontsize=7.5, loc="upper right")

    # --- Hardy–Weinberg: genotype frequencies of N individuals vs p², 2pq, q² ---
    x = np.linspace(0, 1, 101)
    curves = equation.hardy_weinberg(torch.from_numpy(x)).numpy()
    dots = []
    for g, (name, color) in enumerate(GENO):
        ax_hw.plot(x, curves[:, g], color=color, lw=1.2, label=name)
        dots.append(ax_hw.plot([], [], "o", ms=2.5, color=color)[0])
    ax_hw.set_xlim(0, 1)
    ax_hw.set_ylim(0, 1)
    ax_hw.set_xlabel("$p$", labelpad=1)
    ax_hw.set_title(r"Hardy–Weinberg $p^2,\ 2pq,\ q^2$", fontsize=10)
    ax_hw.legend(fontsize=7, loc="upper center", ncol=3, handlelength=1)

    # --- fixation probability of a single new mutant ---
    xs = (4 * N * s_grid).cpu().numpy()
    us, ut = u_sim.cpu().numpy(), u_theory.cpu().numpy()
    fine = torch.linspace(xs.min(), xs.max(), 200, dtype=torch.float64) / (4 * N)
    ax_u.plot(4 * N * fine.numpy(), equation.kimura_fixation(1 / (2 * N), N, fine).numpy(), color="#374151", lw=1.2)
    err = np.sqrt(np.maximum(us * (1 - us), 1e-12) / n_fix)
    ax_u.errorbar(xs, np.maximum(us, 1e-5), yerr=err, fmt="o", ms=4, color="#dc2626", capsize=2)
    ax_u.axhline(1 / (2 * N), color="#9ca3af", ls="--", lw=1)
    ax_u.text(xs.min(), 1.15 / (2 * N), "1/2N", fontsize=8, color="#6b7280")
    ax_u.set_yscale("log")
    lo, hi = max(ut.min() * 0.4, 1e-5), max(us.max(), ut.max()) * 2
    ax_u.set_ylim(lo, hi)
    ax_u.minorticks_off()  # fewer tick artists to redraw in every frame
    ticks = [v for v in (1e-4, 1e-3, 1e-2, 1e-1, 1.0) if lo <= v <= hi]
    ax_u.set_yticks(ticks, [f"{v:g}" for v in ticks])  # plain labels: no mathtext per frame
    ax_u.set_xlabel("$4Ns$", labelpad=1)
    ax_u.set_title("fixation of one new copy: Kimura", fontsize=10)

    def update(step: int):
        for r, line in enumerate(lines):
            line.set_data(t[: step + 1], P[r, : step + 1])
        clock.set_text(f"t = {step}")
        frac = np.histogram(P[:, step], bins=edges)[0] / len(P)
        for bar, f in zip(bars, frac):
            bar.set_height(f)
        fixed, lost = (P[:, step] == 1).mean(), (P[:, step] == 0).mean()
        fate_text.set_text(f"fixed {fixed:.0%}   lost {lost:.0%}")
        h_line.set_data(t[: step + 1], H[: step + 1] / H[0])
        for g in range(3):
            dots[g].set_data(P[: geno.shape[0], step], geno[:, step, g])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(G + 1, 40), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'N': 50,
    'p0': 0.5,
    's': 0.0,
    'generations': 200,
    'replicates': 2000,
    'fix_replicates': 20000,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.wright_fisher_drift")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.wright_fisher_drift.main")
problem_dir = Path("problems/bioinformatics/wright_fisher_drift")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
